# 🎯 CS2 Bot Training - Обучение на топовых матчах
## Сжатые демки с HLTV для экономии места

**Что делает:**
1. Скачивает демки топовых команд с HLTV на твоём ПК
2. Сжимает их в ZIP архив (экономия ~30-50% места)
3. В Colab автоматически распаковывает и парсит
4. Обучает AI на про-игре

**Требования:**
- GPU Runtime (Runtime → Change runtime type → T4 GPU)
- Сжатый архив демок на Google Drive

**Преимущества:**
- Меньше места на Drive (~10-15 GB вместо 20 GB)
- Быстрее загружается в Colab
- Демки с HLTV (без блокировок)

## 🔧 Шаг 1: Проверка GPU

In [ ]:
# Проверка GPU (должен быть Tesla T4 или другой)
!nvidia-smi

## 📦 Шаг 2: Установка библиотек

In [ ]:
# Установка всех необходимых библиотек (~2 минуты)
!pip install -q requests awpy pandas pyarrow tqdm torch zstandard

print("✅ Библиотеки установлены!")

## 📥 Шаг 3: Клонирование репозитория

In [ ]:
# Удаляем старые копии если есть
!rm -rf /content/cs2-bot-training

# Переходим в корень
%cd /content

# Клонируем репозиторий
!git clone https://github.com/nem1k9/cs2-bot-training.git

# Переходим в папку scripts (там все .py файлы)
%cd cs2-bot-training/scripts

# Проверяем что файлы есть
print("\n✅ Файлы проекта:")
!ls -la

## 🎮 Шаг 4: Загрузка демок

**Два варианта:**

**Вариант 1: ZIP архив** (для HLTV демок)
- Установи `USE_ZIP = True`
- Укажи путь к архиву в `DRIVE_ZIP_PATH`

**Вариант 2: Папка с .zst файлами** (для FACEIT демок) ← ИСПОЛЬЗУЙ ЭТОТ!
- Установи `USE_ZIP = False`
- Укажи путь к папке в `DRIVE_FOLDER_PATH`
- Colab автоматически распакует .zst файлы

### 📥 Подключение Drive и копирование демок

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключен!")

In [ ]:
import os
import zipfile
import zstandard as zstd
import glob
from tqdm import tqdm

# ========== НАСТРОЙКИ ==========
# Вариант 1: ZIP архив
DRIVE_ZIP_PATH = "/content/drive/MyDrive/cs2_demos_compressed.zip"

# Вариант 2: Папка с .zst файлами (для FACEIT демок)
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/CS2_Demos_Donk_Wins"

# Выбери что использовать
USE_ZIP = False  # True = ZIP архив, False = папка с .zst

PLAYER = "donk"  # ← Имя игрока для парсинга
# ===============================

print(f"📦 Распаковываем демки...\n")

# Создаём папку для демок
!mkdir -p ./demos

if USE_ZIP:
    # ========== ВАРИАНТ 1: ZIP АРХИВ ==========
    if not os.path.exists(DRIVE_ZIP_PATH):
        print(f"❌ ОШИБКА: Архив не найден!")
        print(f"📁 Ожидаемый путь: {DRIVE_ZIP_PATH}")
    else:
        zip_size_gb = os.path.getsize(DRIVE_ZIP_PATH) / 1e9
        print(f"✅ Архив найден: {zip_size_gb:.2f} GB")
        print(f"📦 Распаковываем...\n")
        
        with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
            file_list = [f for f in zip_ref.namelist() if f.endswith('.dem')]
            print(f"📊 Файлов в архиве: {len(file_list)}")
            
            for file in tqdm(file_list, desc="Распаковка"):
                zip_ref.extract(file, './demos/')
        
        print(f"\n✅ Распаковано {len(file_list)} демок!")
else:
    # ========== ВАРИАНТ 2: ПАПКА С .ZST ФАЙЛАМИ ==========
    if not os.path.exists(DRIVE_FOLDER_PATH):
        print(f"❌ ОШИБКА: Папка не найдена!")
        print(f"📁 Ожидаемый путь: {DRIVE_FOLDER_PATH}")
        print(f"\n💡 РЕШЕНИЕ:")
        print(f"   1. Запусти на ПК: python download_donk_faceit_wins.py")
        print(f"   2. Загрузи папку demos на Google Drive")
        print(f"   3. Укажи правильный путь в DRIVE_FOLDER_PATH")
    else:
        # Ищем .zst файлы
        zst_files = glob.glob(f"{DRIVE_FOLDER_PATH}/*.zst")
        dem_files = glob.glob(f"{DRIVE_FOLDER_PATH}/*.dem")
        
        print(f"✅ Папка найдена!")
        print(f"📊 Найдено файлов:")
        print(f"   .zst (сжатые): {len(zst_files)}")
        print(f"   .dem (распакованные): {len(dem_files)}\n")
        
        # Распаковываем .zst файлы
        if zst_files:
            print(f"📦 Распаковываем .zst файлы...\n")
            dctx = zstd.ZstdDecompressor()
            
            for zst_file in tqdm(zst_files, desc="Распаковка"):
                filename = os.path.basename(zst_file)
                dem_filename = filename.replace('.zst', '')
                output_path = f"./demos/{dem_filename}"
                
                try:
                    with open(zst_file, 'rb') as ifh, open(output_path, 'wb') as ofh:
                        dctx.copy_stream(ifh, ofh)
                except Exception as e:
                    print(f"\n❌ Ошибка распаковки {filename}: {e}")
            
            print(f"\n✅ Распаковано {len(zst_files)} файлов!")
        
        # Копируем уже распакованные .dem файлы
        if dem_files:
            print(f"\n📋 Копируем {len(dem_files)} .dem файлов...")
            !cp {DRIVE_FOLDER_PATH}/*.dem ./demos/ 2>/dev/null || echo "Нет .dem файлов"

# Проверяем результат
print("\n📊 Демки в ./demos/:")
!ls -lh ./demos/*.dem | wc -l
!ls -lh ./demos/*.dem | head -5

print(f"\n✅ Демки готовы к парсингу!")

## 🔍 Шаг 5: Парсинг демок

**Что отслеживается:**
- Движение (позиция, скорость, присед)
- Прицеливание (yaw, pitch, скоп)
- Стрельба (выстрелы, перезарядка)
- Гранаты (флешки, смоки, HE, молотовы)
- Тактика (позиционирование, дистанция до врагов)

**Примечание:** Если указан PLAYER, парсятся только его действия. Если нет - все игроки.

In [ ]:
from parse_demos import build_dataset

print(f"🔄 Парсим демки...")
if PLAYER:
    print(f"🎯 Фокус на игроке: {PLAYER}")
else:
    print(f"🎯 Парсим всех игроков")
    
print(f"⏱️  Это займёт ~10-30 минут...\n")

# Парсим демки (с фильтром по игроку если указан)
dataset = build_dataset(
    demo_dir="./demos",
    out_path="./dataset.parquet",
    target_player=PLAYER if PLAYER else None
)

if PLAYER:
    print(f"\n✅ Датасет для {PLAYER} готов: {len(dataset):,} тиков")
else:
    print(f"\n✅ Датасет готов: {len(dataset):,} тиков")
    
print(f"📊 Размер датасета: {dataset.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 🚀 Шаг 6: Обучение модели

**Время обучения:** ~2-4 часа

**ВАЖНО:** Не закрывай вкладку браузера!

In [ ]:
from train import train

print(f"🚀 Начинаем обучение модели на стиле {PLAYER}...")
print(f"⏱️  Это займёт ~2-4 часа...\n")

train()

print(f"\n✅ Обучение завершено!")
print(f"🎯 Модель обучена копировать стиль игры {PLAYER}!")

## 💾 Шаг 7: Сохранение результатов на Google Drive

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Создаём папку для результатов
!mkdir -p /content/drive/MyDrive/cs2_bot_results

# Сохраняем модели и датасет
print("💾 Сохраняем результаты на Google Drive...\n")

!cp cs2bot_final.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ cs2bot_final.pt сохранён" || echo "⚠️  cs2bot_final.pt не найден"
!cp checkpoint.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ checkpoint.pt сохранён" || echo "⚠️  checkpoint.pt не найден"
!cp dataset.parquet /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ dataset.parquet сохранён" || echo "⚠️  dataset.parquet не найден"

print("\n✅ Результаты сохранены на Google Drive в папке cs2_bot_results!")
print("\n📁 Что сохранено:")
!ls -lh /content/drive/MyDrive/cs2_bot_results/

## 📥 Шаг 8: Скачать модель на компьютер (опционально)

In [ ]:
# Скачать модель прямо в браузер
from google.colab import files

print("📥 Скачиваем модель...")
files.download('cs2bot_final.pt')
files.download('checkpoint.pt')

print("✅ Модель скачана!")

## 🎯 Готово!

**Что получилось:**
- ✅ Скачаны демки топовых команд с HLTV
- ✅ Сжаты в ZIP архив (экономия места ~30-50%)
- ✅ Распакованы в Colab
- ✅ Распарсены все действия игроков
- ✅ Обучена модель на про-игре
- ✅ Результаты сохранены на Google Drive

**Модель умеет:**
- Двигаться как про-игроки
- Целиться и стрелять
- Использовать гранаты
- Принимать тактические решения

**Это AI обученный на топовых матчах!** 🔥